In [ ]:
!pip -q install kagglehub pennylane scikit-learn pandas matplotlib seaborn

In [ ]:
import os
import time
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim

import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset

import kagglehub
import pennylane as qml

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report
)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
IMAGE_SIZE = 64
BATCH_SIZE = 16
MAX_EPOCHS = 100
LEARNING_RATE = 0.001

SPLITS = {
    "80-20": 0.20,
    "60-40": 0.40,
    "90-10": 0.10
}

SEEDS = [42, 123, 2026]
DEVICE = torch.device("cpu")

print("PyTorch:", torch.__version__)
print("PennyLane:", qml.__version__)
print("Device:", DEVICE)
print("Maximum epochs per run:", MAX_EPOCHS)

PyTorch: 2.14.1+cpu
PennyLane: 0.45.1
Device: cpu
Maximum epochs per run: 100


In [ ]:
dataset_path = kagglehub.dataset_download(
    "pankajsthakre/mangoleafds2025"
)

print("Downloaded dataset location:")
print(dataset_path)

Download already complete (16626493255 bytes).
Extracting files...
Downloaded dataset location:
/root/.cache/kagglehub/datasets/pankajsthakre/mangoleafds2025/versions/1


In [ ]:
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def find_imagefolder_root(base_path):
    candidates = []

    for root, dirs, files in os.walk(base_path):
        class_dirs = []
        for folder in dirs:
            folder_path = os.path.join(root, folder)
            try:
                has_image = any(
                    name.lower().endswith(IMAGE_EXTENSIONS)
                    for name in os.listdir(folder_path)
                )
                if has_image:
                    class_dirs.append(folder)
            except (PermissionError, OSError):
                pass

        if len(class_dirs) >= 2:
            candidates.append((root, len(class_dirs)))

    if not candidates:
        raise FileNotFoundError(
            "Could not find a directory containing at least two class "
            "subfolders with images. Inspect dataset_path with os.walk "
            "and set DATA_ROOT manually."
        )

    candidates.sort(key=lambda item: item[1], reverse=True)
    return candidates[0][0]

DATA_ROOT = find_imagefolder_root(dataset_path)
print("Detected DATA_ROOT:", DATA_ROOT)
print("Immediate folders:", os.listdir(DATA_ROOT)[:30])

Detected DATA_ROOT: /root/.cache/kagglehub/datasets/pankajsthakre/mangoleafds2025/versions/1/original_dataset
Immediate folders: ['LEAF BLIGHT', 'HEALTHY', 'INSECT DAMAGE WEBBER', 'GALL MILDGE DAMAGE', 'ANTHRECNOSE', 'DIEBACK']


## 5. Load dataset and define image transformations

In [ ]:
train_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ToTensor(),
    T.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

test_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])
train_full = ImageFolder(DATA_ROOT, transform=train_transform)
test_full = ImageFolder(DATA_ROOT, transform=test_transform)

if train_full.classes != test_full.classes:
    raise ValueError("Class lists differ between training and evaluation datasets.")

if train_full.samples != test_full.samples:
    raise ValueError("Image sample lists differ between dataset objects.")

classes = train_full.classes
labels = np.array(train_full.targets)
num_classes = len(classes)
dataset_size = len(train_full)

print("Number of images:", dataset_size)
print("Number of classes:", num_classes)
print("Class names:", classes)
print("\nImages per class:")
print(pd.Series(labels).map(dict(enumerate(classes))).value_counts())

Number of images: 6640
Number of classes: 6
Class names: ['ANTHRECNOSE', 'DIEBACK', 'GALL MILDGE DAMAGE', 'HEALTHY', 'INSECT DAMAGE WEBBER', 'LEAF BLIGHT']

Images per class:
HEALTHY                 1950
INSECT DAMAGE WEBBER    1136
DIEBACK                 1084
LEAF BLIGHT              949
ANTHRECNOSE              886
GALL MILDGE DAMAGE       635
Name: count, dtype: int64


In [ ]:
class FeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.projection = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 4),
            nn.Tanh()
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.projection(x)
        return x


class CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.feature_extractor = FeatureExtractor()

        self.classifier = nn.Sequential(
            nn.Linear(4, 16),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(16, num_classes)
        )

    def forward(self, x):
        features = self.feature_extractor(x)
        return self.classifier(features)

## 7. Define the 4-qubit quantum circuit and hybrid QCNN

The circuit encodes four classical features into four qubits, applies trainable entangling operations, then uses a small pooling stage. It returns two expectation values to the classical classifier.

This is a compact **QCNN-inspired hybrid circuit**, not a claim that it implements every feature of a canonical QCNN architecture.

In [ ]:
N_QUBITS = 4
quantum_device = qml.device("default.qubit", wires=N_QUBITS)

@qml.qnode(
    quantum_device,
    interface="torch",
    diff_method="backprop"
)
def quantum_circuit(inputs, conv_weights, pool_weights):

    for i in range(N_QUBITS):
        qml.RY(inputs[i], wires=N_QUBITS - i - 1)
        qml.RZ(inputs[i], wires=N_QUBITS - i - 1)
    pairs = [(0, 1), (2, 3), (1, 2), (3, 0)]

    for j, (a, b) in enumerate(pairs):
        qml.RY(conv_weights[j, 0], wires=a)
        qml.RZ(conv_weights[j, 1], wires=b)
        qml.CNOT(wires=[a, b])
        qml.RY(conv_weights[j, 2], wires=b)
        qml.CNOT(wires=[a, b])
    qml.CNOT(wires=[1, 0])
    qml.CRY(pool_weights[0], wires=[0, 1])

    qml.CNOT(wires=[3, 2])
    qml.CRY(pool_weights[1], wires=[2, 3])

    return [
        qml.expval(qml.PauliZ(0)),
        qml.expval(qml.PauliZ(2))
    ]


weight_shapes = {
    "conv_weights": (4, 3),
    "pool_weights": (2,)
}


class QCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.feature_extractor = FeatureExtractor()

        self.quantum_layer = qml.qnn.TorchLayer(
            quantum_circuit,
            weight_shapes
        )

        self.classifier = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(16, num_classes)
        )

    def forward(self, x):
        features = self.feature_extractor(x)
        quantum_features = self.quantum_layer(angles)

        return self.classifier(quantum_features)

## 8. Define evaluation metrics

In [ ]:
def evaluate(model, loader):
    model.eval()

    actual = []
    predicted = []

    start_time = time.perf_counter()

    with torch.no_grad():
        for images, batch_labels in loader:
            images = images.to(DEVICE)

            outputs = model(images)
            predictions = outputs.argmax(dim=1)

            actual.extend(batch_labels.numpy().tolist())
            predicted.extend(predictions.cpu().numpy().tolist())

    elapsed = time.perf_counter() - start_time

    metrics = {
        "Accuracy": accuracy_score(actual, predicted) * 100,
        "Precision": precision_score(
            actual, predicted, average="macro", zero_division=0
        ) * 100,
        "Recall": recall_score(
            actual, predicted, average="macro", zero_division=0
        ) * 100,
        "Macro F1": f1_score(
            actual, predicted, average="macro", zero_division=0
        ) * 100,
        "Balanced Accuracy": balanced_accuracy_score(
            actual, predicted
        ) * 100,
        "Testing Time (sec)": elapsed,
        "Inference (ms/image)": (
            elapsed / len(actual) * 1000 if len(actual) else np.nan
        )
    }

    return metrics, actual, predicted

In [ ]:
def train_model(model, train_loader, val_loader, class_weights):
    model.to(DEVICE)

    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(DEVICE)
    )

    optimizer = optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    best_val_f1 = -1.0
    best_weights = copy.deepcopy(model.state_dict())

    history = []
    cumulative_training_time = 0.0
    cumulative_validation_time = 0.0

    for epoch in range(MAX_EPOCHS):
        model.train()
        total_loss = 0.0
        train_start = time.perf_counter()

        for images, batch_labels in train_loader:
            images = images.to(DEVICE)
            batch_labels = batch_labels.to(DEVICE)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, batch_labels)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        epoch_training_time = time.perf_counter() - train_start
        cumulative_training_time += epoch_training_time
        val_start = time.perf_counter()
        val_metrics, _, _ = evaluate(model, val_loader)
        epoch_validation_time = time.perf_counter() - val_start
        cumulative_validation_time += epoch_validation_time

        average_loss = total_loss / max(1, len(train_loader))
        val_accuracy = val_metrics["Accuracy"]
        val_f1 = val_metrics["Macro F1"]

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_weights = copy.deepcopy(model.state_dict())

        history.append({
            "Epoch": epoch + 1,
            "Training Loss": average_loss,
            "Validation Accuracy": val_accuracy,
            "Validation F1": val_f1,
            "Epoch Training Time (sec)": epoch_training_time,
            "Cumulative Training Time (sec)": cumulative_training_time,
            "Cumulative Validation Time (sec)": cumulative_validation_time
        })

        print(
            f"Epoch {epoch + 1:03d}/{MAX_EPOCHS} | "
            f"Loss: {average_loss:.4f} | "
            f"Val Accuracy: {val_accuracy:.2f}% | "
            f"Val Macro F1: {val_f1:.2f}% | "
            f"Train time: {epoch_training_time:.2f}s"
        )
    model.load_state_dict(best_weights)

    return (
        model,
        pd.DataFrame(history),
        cumulative_training_time,
        cumulative_validation_time
    )

In [ ]:
results = []
histories = {}
predictions_store = {}

all_indices = np.arange(dataset_size)

for split_name, test_size in SPLITS.items():

    for seed in SEEDS:

        print("\n" + "=" * 75)
        print(f"STARTING SPLIT: {split_name} | SEED: {seed}")
        print("=" * 75)
        train_pool_indices, test_indices = train_test_split(
            all_indices,
            test_size=test_size,
            random_state=seed,
            stratify=labels
        )
        train_indices, val_indices = train_test_split(
            train_pool_indices,
            test_size=0.15,
            random_state=seed,
            stratify=labels[train_pool_indices]
        )

        train_data = Subset(train_full, train_indices)
        val_data = Subset(test_full, val_indices)
        test_data = Subset(test_full, test_indices)

        class_counts = np.bincount(
            labels[train_indices],
            minlength=num_classes
        )

        if np.any(class_counts == 0):
            raise ValueError(
                "A class is missing from a training fold. "
                "Check class counts or reduce the number of splits."
            )

        class_weight_values = (
            len(train_indices) / (num_classes * class_counts)
        )

        class_weights = torch.tensor(
            class_weight_values,
            dtype=torch.float32
        )

        val_loader = DataLoader(
            val_data,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=0
        )

        test_loader = DataLoader(
            test_data,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=0
        )

        for model_name in ["CNN", "QCNN"]:

            print("\n" + "-" * 60)
            print(f"TRAINING {model_name} | {split_name} | seed={seed}")
            print("-" * 60)
            set_seed(seed)
            train_generator = torch.Generator()
            train_generator.manual_seed(seed)
            train_loader = DataLoader(
                train_data,
                batch_size=BATCH_SIZE,
                shuffle=True,
                num_workers=0,
                generator=train_generator
            )

            if model_name == "CNN":
                model = CNN(num_classes)
            else:
                model = QCNN(num_classes)

            (
                model,
                history_df,
                training_time,
                validation_time
            ) = train_model(
                model,
                train_loader,
                val_loader,
                class_weights
            )
            test_metrics, actual, predicted = evaluate(
                model,
                test_loader
            )

            parameter_count = sum(
                parameter.numel()
                for parameter in model.parameters()
                if parameter.requires_grad
            )

            key = (split_name, seed, model_name)

            histories[key] = history_df

            predictions_store[key] = {
                "actual": actual,
                "predicted": predicted
            }

            results.append({
                "Split": split_name,
                "Seed": seed,
                "Model": model_name,
                "Epochs Trained": len(history_df),
                "Training Time (sec)": training_time,
                "Validation Time (sec)": validation_time,
                "Trainable Parameters": parameter_count,
                **test_metrics
            })

            print("\nTEST RESULTS")
            for metric_name, metric_value in test_metrics.items():
                print(f"{metric_name}: {metric_value:.4f}")

            print("Training time (sec):", round(training_time, 2))
            print("Epochs completed:", len(history_df))

        pd.DataFrame(results).to_csv(
            "cnn_qcnn_intermediate_results.csv",
            index=False
        )

results_df = pd.DataFrame(results)

print("\nALL 18 MODEL RUNS COMPLETED.")
print(results_df.head())


STARTING SPLIT: 80-20 | SEED: 42

------------------------------------------------------------
TRAINING CNN | 80-20 | seed=42
------------------------------------------------------------
Epoch 001/100 | Loss: 1.5695 | Val Accuracy: 45.04% | Val Macro F1: 38.89% | Train time: 870.72s
Epoch 002/100 | Loss: 1.2788 | Val Accuracy: 55.58% | Val Macro F1: 46.68% | Train time: 886.26s
Epoch 003/100 | Loss: 1.2232 | Val Accuracy: 59.85% | Val Macro F1: 48.89% | Train time: 857.96s
Epoch 004/100 | Loss: 1.1797 | Val Accuracy: 61.73% | Val Macro F1: 56.25% | Train time: 872.16s
Epoch 005/100 | Loss: 1.1224 | Val Accuracy: 61.48% | Val Macro F1: 59.14% | Train time: 882.45s


In [ ]:
display(
    results_df.sort_values(["Split", "Seed", "Model"]).round(3)
)

results_df.to_csv("cnn_qcnn_all_run_results.csv", index=False)

In [ ]:
metric_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "Macro F1",
    "Balanced Accuracy",
    "Training Time (sec)",
    "Testing Time (sec)",
    "Inference (ms/image)",
    "Trainable Parameters"
]

summary_df = (
    results_df
    .groupby(["Split", "Model"])[metric_columns]
    .agg(["mean", "std"])
    .round(3)
)

display(summary_df)
summary_df.to_csv("cnn_qcnn_mean_std_summary.csv")

In [ ]:
epoch_comparison_rows = []

for split_name in SPLITS:
    for seed in SEEDS:

        cnn_history = histories[(split_name, seed, "CNN")]
        qcnn_history = histories[(split_name, seed, "QCNN")]

        cnn_best_val_accuracy = cnn_history[
            "Validation Accuracy"
        ].max()

        target_accuracy = max(
            0.0,
            cnn_best_val_accuracy - 1.0
        )

        cnn_reached = cnn_history[
            cnn_history["Validation Accuracy"] >= target_accuracy
        ]

        qcnn_reached = qcnn_history[
            qcnn_history["Validation Accuracy"] >= target_accuracy
        ]

        def first_reach_value(frame, column):
            if frame.empty:
                return np.nan
            return frame.iloc[0][column]

        cnn_epoch = first_reach_value(cnn_reached, "Epoch")
        qcnn_epoch = first_reach_value(qcnn_reached, "Epoch")

        cnn_time = first_reach_value(
            cnn_reached,
            "Cumulative Training Time (sec)"
        )

        qcnn_time = first_reach_value(
            qcnn_reached,
            "Cumulative Training Time (sec)"
        )

        epoch_reduction = np.nan
        if pd.notna(cnn_epoch) and pd.notna(qcnn_epoch) and cnn_epoch > 0:
            epoch_reduction = (
                (cnn_epoch - qcnn_epoch) / cnn_epoch
            ) * 100

        time_reduction = np.nan
        if pd.notna(cnn_time) and pd.notna(qcnn_time) and cnn_time > 0:
            time_reduction = (
                (cnn_time - qcnn_time) / cnn_time
            ) * 100

        epoch_comparison_rows.append({
            "Split": split_name,
            "Seed": seed,
            "Common Validation Target (%)": target_accuracy,
            "CNN Epochs to Target": cnn_epoch,
            "QCNN Epochs to Target": qcnn_epoch,
            "Epoch Reduction (%)": epoch_reduction,
            "CNN Training Time to Target (sec)": cnn_time,
            "QCNN Training Time to Target (sec)": qcnn_time,
            "Training-Time Reduction (%)": time_reduction
        })

epoch_comparison_df = pd.DataFrame(epoch_comparison_rows)

display(epoch_comparison_df.round(3))

epoch_comparison_df.to_csv(
    "cnn_qcnn_epoch_convergence_comparison.csv",
    index=False
)

In [ ]:
convergence_summary = (
    epoch_comparison_df
    .groupby("Split")
    .agg({
        "Common Validation Target (%)": "mean",
        "CNN Epochs to Target": "mean",
        "QCNN Epochs to Target": "mean",
        "Epoch Reduction (%)": "mean",
        "CNN Training Time to Target (sec)": "mean",
        "QCNN Training Time to Target (sec)": "mean",
        "Training-Time Reduction (%)": "mean"
    })
    .round(3)
)

display(convergence_summary)
convergence_summary.to_csv(
    "cnn_qcnn_convergence_summary.csv"
)

In [ ]:
for split_name in SPLITS:

    plt.figure(figsize=(10, 6))

    for model_name in ["CNN", "QCNN"]:

        seed_histories = [
            histories[(split_name, seed, model_name)]
            for seed in SEEDS
        ]

        accuracy_matrix = np.stack([
            frame["Validation Accuracy"].to_numpy()
            for frame in seed_histories
        ])

        mean_accuracy = accuracy_matrix.mean(axis=0)
        std_accuracy = accuracy_matrix.std(axis=0)

        epochs = np.arange(1, MAX_EPOCHS + 1)

        plt.plot(
            epochs,
            mean_accuracy,
            label=model_name
        )

        plt.fill_between(
            epochs,
            mean_accuracy - std_accuracy,
            mean_accuracy + std_accuracy,
            alpha=0.15
        )

    plt.xlabel("Epoch")
    plt.ylabel("Validation Accuracy (%)")
    plt.title(f"CNN vs QCNN Validation Accuracy — {split_name}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
for split_name in SPLITS:

    plt.figure(figsize=(10, 6))

    for model_name in ["CNN", "QCNN"]:

        seed_histories = [
            histories[(split_name, seed, model_name)]
            for seed in SEEDS
        ]

        loss_matrix = np.stack([
            frame["Training Loss"].to_numpy()
            for frame in seed_histories
        ])

        mean_loss = loss_matrix.mean(axis=0)
        epochs = np.arange(1, MAX_EPOCHS + 1)

        plt.plot(
            epochs,
            mean_loss,
            label=model_name
        )

    plt.xlabel("Epoch")
    plt.ylabel("Training Loss")
    plt.title(f"CNN vs QCNN Training Loss — {split_name}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
for metric in ["Accuracy", "Macro F1"]:

    plot_data = (
        results_df
        .groupby(["Split", "Model"])[metric]
        .agg(["mean", "std"])
        .reset_index()
    )

    plt.figure(figsize=(9, 5))

    sns.barplot(
        data=plot_data,
        x="Split",
        y="mean",
        hue="Model",
        errorbar=None)
    plt.ylim(0, 100)
    for x_index, split_name in enumerate(SPLITS.keys()):
        for model_index, model_name in enumerate(["CNN", "QCNN"]):
            row = plot_data[
                (plot_data["Split"] == split_name) &
                (plot_data["Model"] == model_name)
            ]

            if not row.empty:
                offset = (-0.2 if model_name == "CNN" else 0.2)
                mean_value = row["mean"].iloc[0]
                std_value = row["std"].iloc[0]
                if pd.notna(std_value):
                    plt.errorbar(
                        x=x_index + offset,
                        y=mean_value,
                        yerr=std_value,
                        fmt="none",
                        capsize=4
                    )

    plt.ylabel(f"{metric} (%)")
    plt.xlabel("Train-Test Split")
    plt.title(f"CNN vs QCNN — {metric}")
    plt.grid(axis="y")
    plt.tight_layout()
    plt.show()

In [ ]:
for split_name in SPLITS:

    for model_name in ["CNN", "QCNN"]:

        key = (split_name, SEEDS[0], model_name)

        actual = predictions_store[key]["actual"]
        predicted = predictions_store[key]["predicted"]

        print("\n", "=" * 65)
        print(f"{model_name} | {split_name} | seed={SEEDS[0]}")
        print("=" * 65)

        print(
            classification_report(
                actual,
                predicted,
                labels=list(range(num_classes)),
                target_names=classes,
                zero_division=0
            )
        )

        cm = confusion_matrix(
            actual,
            predicted,
            labels=list(range(num_classes))
        )

        plt.figure(figsize=(8, 6))
        sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=classes,
            yticklabels=classes
        )
        plt.xlabel("Predicted label")
        plt.ylabel("True label")
        plt.title(f"Confusion Matrix — {model_name} — {split_name}")
        plt.tight_layout()
        plt.show()

In [ ]:
results_df.to_csv("cnn_qcnn_all_run_results.csv", index=False)
summary_df.to_csv("cnn_qcnn_mean_std_summary.csv")
epoch_comparison_df.to_csv(
    "cnn_qcnn_epoch_convergence_comparison.csv",
    index=False
)
convergence_summary.to_csv(
    "cnn_qcnn_convergence_summary.csv"
)

history_rows = []

for (split_name, seed, model_name), history_df in histories.items():
    temp = history_df.copy()
    temp["Split"] = split_name
    temp["Seed"] = seed
    temp["Model"] = model_name
    history_rows.append(temp)

all_histories_df = pd.concat(history_rows, ignore_index=True)

all_histories_df.to_csv(
    "cnn_qcnn_all_epoch_histories.csv",
    index=False
)

print("Saved all result and epoch-history CSV files.")
print("Files:", [
    "cnn_qcnn_all_run_results.csv",
    "cnn_qcnn_mean_std_summary.csv",
    "cnn_qcnn_epoch_convergence_comparison.csv",
    "cnn_qcnn_convergence_summary.csv",
    "cnn_qcnn_all_epoch_histories.csv"
])

In [ ]:
class ClassicalBottleneck(nn.Module):
    """Ablation baseline: replace the quantum layer with a classical 4-to-2 layer."""
    def __init__(self, num_classes):
        super().__init__()
        self.feature_extractor = FeatureExtractor()
        self.classical_projection = nn.Linear(4, 2)
        self.classifier = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(16, num_classes)
        )

    def forward(self, x):
        features = self.feature_extractor(x)
        reduced_features = torch.tanh(self.classical_projection(features))
        return self.classifier(reduced_features)


baseline_results = []
baseline_predictions = {}

for split_name, test_size in SPLITS.items():
    for seed in SEEDS:
        print("\\n" + "=" * 75)
        print(f"CLASSICAL BOTTLENECK | {split_name} | seed={seed}")
        print("=" * 75)

        train_pool_indices, test_indices = train_test_split(
            all_indices,
            test_size=test_size,
            random_state=seed,
            stratify=labels
        )
        train_indices, val_indices = train_test_split(
            train_pool_indices,
            test_size=0.15,
            random_state=seed,
            stratify=labels[train_pool_indices]
        )

        train_data = Subset(train_full, train_indices)
        val_data = Subset(test_full, val_indices)
        test_data = Subset(test_full, test_indices)

        class_counts = np.bincount(labels[train_indices],
                                   minlength=num_classes)
        if np.any(class_counts == 0):
            raise ValueError("A class is missing from a training fold.")

        class_weight_values = len(train_indices) / (num_classes * class_counts)
        class_weights = torch.tensor(class_weight_values, dtype=torch.float32)

        train_generator = torch.Generator()
        train_generator.manual_seed(seed)
        train_loader = DataLoader(
            train_data, batch_size=BATCH_SIZE, shuffle=True,
            num_workers=0, generator=train_generator
        )
        val_loader = DataLoader(
            val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
        )
        test_loader = DataLoader(
            test_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
        )

        set_seed(seed)
        model = ClassicalBottleneck(num_classes)

        model, history_df, training_time, validation_time = train_model(
            model, train_loader, val_loader, class_weights
        )
        test_metrics, actual, predicted = evaluate(model, test_loader)

        parameter_count = sum(
            p.numel() for p in model.parameters() if p.requires_grad
        )
        baseline_predictions[(split_name, seed)] = {
            "actual": actual, "predicted": predicted
        }
        baseline_results.append({
            "Split": split_name,
            "Seed": seed,
            "Model": "Classical Bottleneck",
            "Epochs Trained": len(history_df),
            "Training Time (sec)": training_time,
            "Validation Time (sec)": validation_time,
            "Trainable Parameters": parameter_count,
            **test_metrics
        })

        pd.DataFrame(baseline_results).to_csv(
            "classical_bottleneck_intermediate_results.csv", index=False
        )

baseline_df = pd.DataFrame(baseline_results)
display(baseline_df.sort_values(["Split", "Seed"]).round(3))
baseline_df.to_csv("cnn_qcnn_classical_ablation_results.csv", index=False)

print("\\nCNN/QCNN/ablation runs:", len(results_df) + len(baseline_df))


## 23. Compare CNN, QCNN and classical ablation

This table puts the three model families side by side. Use mean and standard deviation across the same seeds. The classical bottleneck is an ablation check, not proof that quantum computing is faster.


In [ ]:
all_model_results_df = pd.concat(
    [results_df, baseline_df], ignore_index=True
)

extended_metric_columns = [
    "Accuracy", "Precision", "Recall", "Macro F1",
    "Balanced Accuracy", "Training Time (sec)",
    "Inference (ms/image)", "Trainable Parameters"
]

three_model_summary = (
    all_model_results_df
    .groupby(["Split", "Model"])[extended_metric_columns]
    .agg(["mean", "std"])
    .round(3)
)

display(three_model_summary)
three_model_summary.to_csv("cnn_qcnn_three_model_summary.csv")

for metric in ["Accuracy", "Macro F1"]:
    plot_data = (
        all_model_results_df
        .groupby(["Split", "Model"])[metric]
        .mean()
        .reset_index()
    )
    plt.figure(figsize=(10, 5))
    sns.barplot(data=plot_data, x="Split", y=metric, hue="Model")
    plt.ylabel(f"{metric} (%)")
    plt.title(f"Three-model comparison: {metric}")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
from scipy.stats import ttest_rel, t

paired_stats_rows = []

for split_name in SPLITS:
    split_rows = results_df[results_df["Split"] == split_name]
    pivot = split_rows.pivot(index="Seed", columns="Model", values="Accuracy")
    paired_diff = (pivot["QCNN"] - pivot["CNN"]).dropna().to_numpy()

    n = len(paired_diff)
    mean_diff = float(np.mean(paired_diff)) if n else np.nan
    std_diff = float(np.std(paired_diff, ddof=1)) if n > 1 else np.nan

    if n > 1:
        standard_error = std_diff / np.sqrt(n)
        critical_value = t.ppf(0.975, df=n - 1)
        ci_low = mean_diff - critical_value * standard_error
        ci_high = mean_diff + critical_value * standard_error
        test_result = ttest_rel(pivot["QCNN"], pivot["CNN"])
        p_value = float(test_result.pvalue)
    else:
        ci_low, ci_high, p_value = np.nan, np.nan, np.nan

    paired_stats_rows.append({
        "Split": split_name,
        "Number of paired seeds": n,
        "Mean QCNN-CNN accuracy difference (percentage points)": mean_diff,
        "Std of paired differences": std_diff,
        "95% CI lower": ci_low,
        "95% CI upper": ci_high,
        "Paired t-test p-value": p_value
    })

paired_stats_df = pd.DataFrame(paired_stats_rows)
display(paired_stats_df.round(4))
paired_stats_df.to_csv("cnn_qcnn_paired_statistical_tests.csv", index=False)

print("Interpretation: positive mean difference favours QCNN; negative favours CNN.")
print("A confidence interval crossing zero means the observed direction is uncertain at this level.")


In [ ]:
from sklearn.metrics import precision_recall_fscore_support

per_class_rows = []

for split_name in SPLITS:
    for seed in SEEDS:
        for model_name in ["CNN", "QCNN", "Classical Bottleneck"]:
            if model_name == "Classical Bottleneck":
                stored = baseline_predictions[(split_name, seed)]
            else:
                stored = predictions_store[(split_name, seed, model_name)]

            precision_values, recall_values, f1_values, support_values = (
                precision_recall_fscore_support(
                    stored["actual"],
                    stored["predicted"],
                    labels=list(range(num_classes)),
                    zero_division=0
                )
            )

            for class_index, class_name in enumerate(classes):
                per_class_rows.append({
                    "Split": split_name,
                    "Seed": seed,
                    "Model": model_name,
                    "Class": class_name,
                    "Precision (%)": precision_values[class_index] * 100,
                    "Recall (%)": recall_values[class_index] * 100,
                    "F1 (%)": f1_values[class_index] * 100,
                    "Support": support_values[class_index]
                })

per_class_df = pd.DataFrame(per_class_rows)
per_class_summary_df = (
    per_class_df
    .groupby(["Split", "Model", "Class"])[
        ["Precision (%)", "Recall (%)", "F1 (%)"]
    ]
    .agg(["mean", "std"])
    .round(3)
)

display(per_class_summary_df)
per_class_df.to_csv("cnn_qcnn_per_class_all_seeds.csv", index=False)
per_class_summary_df.to_csv("cnn_qcnn_per_class_mean_std.csv")


In [ ]:
import hashlib
from collections import defaultdict

def sha256_file(file_path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(file_path, "rb") as file:
        for chunk in iter(lambda: file.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

hash_to_indices = defaultdict(list)
for sample_index, (file_path, _) in enumerate(train_full.samples):
    hash_to_indices[sha256_file(file_path)].append(sample_index)

duplicate_groups = {
    image_hash: indices
    for image_hash, indices in hash_to_indices.items()
    if len(indices) > 1
}

duplicate_rows = []
for image_hash, indices in duplicate_groups.items():
    duplicate_rows.append({
        "SHA256": image_hash,
        "Number of copies": len(indices),
        "Paths": " | ".join(train_full.samples[i][0] for i in indices),
        "Labels": " | ".join(classes[labels[i]] for i in indices)
    })

duplicate_df = pd.DataFrame(duplicate_rows)
display(duplicate_df if not duplicate_df.empty else
        pd.DataFrame({"Duplicate check": ["No exact duplicate files detected."]}))
duplicate_df.to_csv("dataset_exact_duplicate_groups.csv", index=False)

leakage_rows = []
for split_name, test_size in SPLITS.items():
    for seed in SEEDS:
        train_pool_indices, test_indices = train_test_split(
            all_indices, test_size=test_size, random_state=seed,
            stratify=labels
        )
        train_hashes = {sha256_file(train_full.samples[i][0]) for i in train_pool_indices}
        test_hashes = {sha256_file(train_full.samples[i][0]) for i in test_indices}
        overlap = train_hashes.intersection(test_hashes)
        leakage_rows.append({
            "Split": split_name,
            "Seed": seed,
            "Exact duplicate hashes across train/test": len(overlap),
            "Potential exact-duplicate leakage": bool(overlap)
        })

leakage_df = pd.DataFrame(leakage_rows)
display(leakage_df)
leakage_df.to_csv("dataset_train_test_duplicate_leakage_check.csv", index=False)


In [ ]:

all_model_results_df.to_csv("cnn_qcnn_all_models_all_runs.csv", index=False)
three_model_summary.to_csv("cnn_qcnn_three_model_summary.csv")
paired_stats_df.to_csv("cnn_qcnn_paired_statistical_tests.csv", index=False)
per_class_df.to_csv("cnn_qcnn_per_class_all_seeds.csv", index=False)
per_class_summary_df.to_csv("cnn_qcnn_per_class_mean_std.csv")
duplicate_df.to_csv("dataset_exact_duplicate_groups.csv", index=False)
leakage_df.to_csv("dataset_train_test_duplicate_leakage_check.csv", index=False)

print("Extended experiment complete. New files:")
for filename in [
    "cnn_qcnn_classical_ablation_results.csv",
    "cnn_qcnn_three_model_summary.csv",
    "cnn_qcnn_paired_statistical_tests.csv",
    "cnn_qcnn_per_class_all_seeds.csv",
    "cnn_qcnn_per_class_mean_std.csv",
    "dataset_exact_duplicate_groups.csv",
    "dataset_train_test_duplicate_leakage_check.csv"
]:
    print(" -", filename)
